# Lab 08-11 · Memories and Lookup Tables

Companion to chapter [08-11](../../chapters/08-11-memories-and-programmable-logic.md). Handout: [README.md](README.md).

Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import math
import random
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

## Part 1 · Organizing memory arrays

For each capacity, with 32-bit words, choose the most nearly square array whose row count is a power of 2, and split the address into row and column bits.

In [2]:
def organize(total_bits, word_bits):
    n_addr = int(math.log2(total_bits // word_bits))
    rows = 2 ** (int(math.log2(total_bits)) // 2)          # rows <= columns, both powers of 2
    cols = total_bits // rows
    per_row = cols // word_bits
    return dict(addr=n_addr, rows=rows, cols=cols, per_row=per_row,
                row_bits=int(math.log2(rows)), col_bits=int(math.log2(per_row)))


print(f"{'capacity':>10s} {'addr':>5s} {'rows':>7s} {'cols':>7s} {'words/row':>10s} {'row bits':>9s} {'col bits':>9s}")
for exp in (10, 14, 16, 20, 23, 30):
    o = organize(2 ** exp, 32)
    assert o["row_bits"] + o["col_bits"] == o["addr"]
    print(f"{'2^' + str(exp):>10s} {o['addr']:5d} {o['rows']:7d} {o['cols']:7d} {o['per_row']:10d} {o['row_bits']:9d} {o['col_bits']:9d}")
gb = organize(2 ** 30, 32)

  capacity  addr    rows    cols  words/row  row bits  col bits
      2^10     5      32      32          1         5         0
      2^14     9     128     128          4         7         2
      2^16    11     256     256          8         8         3
      2^20    15    1024    1024         32        10         5
      2^23    18    2048    4096        128        11         7
      2^30    25   32768   32768       1024        15        10


A single 1 Gbit square array would have 32 768-cell wordlines and bitlines, far too long to be fast; real DRAM and SRAM chips divide it into many banks and sub-arrays (Problem 08-11.13).

## Part 2 · Mapping functions onto 4-input LUTs (the chapter's guided exercise)

`shannon_map` splits a function on its top variable until at most $k$ inputs remain, uses one LUT per leaf and one LUT per 2:1 multiplexer, and **shares** identical sub-functions (a cofactor that has already been built is reused). The result is checked by evaluating the LUT network on every input.

**🔮 Predict:** how many 4-input LUTs does a random 8-input function need? Does sharing identical sub-functions help for random functions? For parity?

In [3]:
def shannon_map(tt, n, k=4):
    # tt: truth table as a tuple of 2^n bits (index = input value). Returns (number of LUTs, evaluator).
    built = {}

    def build(t, m):
        if t in built:
            return built[t]
        if m <= k:
            node = ("lut", t)                               # one LUT holds the whole table
        else:
            half = len(t) // 2
            lo, hi = build(t[:half], m - 1), build(t[half:], m - 1)
            node = ("mux", m - 1, lo, hi)                  # one LUT used as a multiplexer on input m - 1
        built[t] = node
        return node

    root = build(tuple(tt), n)

    def ev(node, x):
        if node[0] == "lut":
            return node[1][x % len(node[1])]
        _, var, lo, hi = node
        return ev(hi if (x >> var) & 1 else lo, x)

    return len(built), (lambda x: ev(root, x))


rng = random.Random(811)
counts = []
for trial in range(1000):
    tt = tuple(rng.getrandbits(1) for _ in range(256))
    n_luts, f = shannon_map(tt, 8)
    if trial < 20:
        assert all(f(x) == tt[x] for x in range(256))
    counts.append(n_luts)
print(f"random 8-input functions: average {np.mean(counts):.3f} LUTs (worst case 2^(8-4+1) - 1 = 31), min {min(counts)}, max {max(counts)}")

parity = tuple(bin(x).count("1") % 2 for x in range(256))
p_luts, pf = shannon_map(parity, 8)
assert all(pf(x) == parity[x] for x in range(256))
print(f"8-input parity: {p_luts} LUTs by Shannon expansion with sharing; 3 LUTs as an XOR tree")

random 8-input functions: average 30.998 LUTs (worst case 2^(8-4+1) - 1 = 31), min 30, max 31
8-input parity: 9 LUTs by Shannon expansion with sharing; 3 LUTs as an XOR tree


Random functions almost never repeat a 4-input sub-function (there are 65 536 possibilities), so sharing saves nearly nothing and the count sits at the worst case. Parity is the opposite: its cofactors are only parity and inverted parity of the remaining inputs, so sharing collapses the tree, and a designer who sees the structure does better still with an XOR tree. Real technology mappers search for this kind of structure, which is why a synthesized netlist of real logic needs far fewer LUTs than the worst case.

## Part 3 · ROM or logic?

Compare the ROM size of three functions with the size of a logic implementation from earlier chapters.

In [4]:
cases = [
    ("4 x 4 multiplier", 8, 8, "16 ANDs + 4-bit Dadda tree and adder (08-09)"),
    ("8 x 8 multiplier", 16, 16, "64 ANDs + 35 full adders + 7 half adders + 14-bit adder (08-09)"),
    ("8-bit binary to Gray", 8, 8, "7 XOR gates (08-01)"),
    ("7-segment decoder", 4, 7, "seven 4-input sums of products, minimized as in 08-03"),
]
for name, n_in, n_out, logic in cases:
    print(f"{name:22s} ROM {2 ** n_in * n_out:>9,d} bits   logic: {logic}")

4 x 4 multiplier       ROM     2,048 bits   logic: 16 ANDs + 4-bit Dadda tree and adder (08-09)
8 x 8 multiplier       ROM 1,048,576 bits   logic: 64 ANDs + 35 full adders + 7 half adders + 14-bit adder (08-09)
8-bit binary to Gray   ROM     2,048 bits   logic: 7 XOR gates (08-01)
7-segment decoder      ROM       112 bits   logic: seven 4-input sums of products, minimized as in 08-03


The ROM's size depends only on the number of inputs and outputs; the logic's depends on the function's structure. Small, irregular functions (the 7-segment decoder) suit ROMs and LUTs; large or structured ones (multipliers, code converters) suit logic.

## Part 4 · Budgeting a small FPGA

A wearable feature on an iCE40 UP5K (5280 logic cells, 30 block RAMs of 4 kbit, 4 single-port RAMs of 256 kbit, 8 DSP blocks) needs an audio buffer, a display buffer and filters. Tally the memory and DSP use.

In [5]:
UP5K = dict(ebr_blocks=30, ebr_bits=4096, spram_blocks=4, spram_bits=256 * 1024, dsp=8)
needs = [("audio buffer, 4096 x 16", 4096 * 16), ("display buffer, 320 x 240 x 1", 320 * 240)]
used = 0
for name, bits in needs:
    blocks = math.ceil(bits / UP5K["ebr_bits"])
    used += blocks
    print(f"{name:32s} {bits:7d} bits -> {blocks:2d} block RAMs")
print(f"block RAMs used: {used} of {UP5K['ebr_blocks']}", "(does not fit)" if used > UP5K["ebr_blocks"] else "")
spram_for_display = math.ceil(320 * 240 / UP5K["spram_bits"])
print(f"moving the display buffer to single-port RAM uses {spram_for_display} of {UP5K['spram_blocks']} blocks; "
      f"block RAMs then used: {math.ceil(4096 * 16 / UP5K['ebr_bits'])}")
taps, fs, f_dsp = 64, 16_000, 12e6
print(f"a {taps}-tap filter at {fs} Hz uses {taps * fs / f_dsp:.1%} of one DSP block at {f_dsp / 1e6:.0f} MHz")

audio buffer, 4096 x 16            65536 bits -> 16 block RAMs
display buffer, 320 x 240 x 1      76800 bits -> 19 block RAMs
block RAMs used: 35 of 30 (does not fit)
moving the display buffer to single-port RAM uses 1 of 4 blocks; block RAMs then used: 16
a 64-tap filter at 16000 Hz uses 8.5% of one DSP block at 12 MHz


The two buffers together need 35 block RAMs, more than the 30 available, so one must move: the display buffer fits easily in one single-port RAM (which suits a frame buffer read and written in sequence). Budgeting like this, before writing any RTL, is how FPGA designs are sized.

**Verilog.** Now run `mem_tb.v` (`python tools/run_hdl.py` from the repository root). Its `lut6_from_lut4` module is exactly the 7-LUT Shannon network that `shannon_map` builds for a 6-input function with no shared sub-functions.